# SECAI — Client DB Data Cleaning
Source: `timesheet_output.xlsx` → sheet `All Records` (441 rows × 23 columns)

Screens the 8 key columns for missing values and typos before loading into the client DB.

In [ ]:
import re
import pandas as pd
import numpy as np

FILE_PATH  = "timesheet_output.xlsx"
SHEET_NAME = "All Records"

COLUMNS_TO_CHECK = [
    "Source File",
    "Page",
    "Date",
    "Day",
    "Title",
    "Employee Name",
    "EIN",
    "Actual Hours",
]

df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME, dtype=str)
df.columns = df.columns.str.strip()
for col in df.columns:
    df[col] = df[col].str.strip()

# ── Jinja2-safe styled display helper ───────────────────────────────────────
try:
    import jinja2
    HAS_JINJA2 = True
except ImportError:
    HAS_JINJA2 = False
    print("Note: jinja2 not installed — tables will display without colour highlighting.")
    print("Install with:  pip install jinja2")

def show(tbl, flag_col="Flag"):
    """Display a dataframe with red rows where Flag is non-empty.
    Falls back to plain display if jinja2 is unavailable."""
    if HAS_JINJA2:
        display(tbl.style.apply(
            lambda r: ["background-color: #fdd" if r[flag_col] else "" for _ in r],
            axis=1
        ))
    else:
        display(tbl)

print(f"Loaded  {len(df):,} rows x {len(df.columns)} columns from '{SHEET_NAME}'")

## 1 · Missing value summary

In [ ]:
rows = []
for col in COLUMNS_TO_CHECK:
    if col not in df.columns:
        rows.append({"Column": col, "Missing count": "COLUMN NOT FOUND", "Missing %": "-"})
        continue
    mask = df[col].isna() | df[col].eq("")
    n = int(mask.sum())
    rows.append({"Column": col, "Missing count": n, "Missing %": f"{n/len(df)*100:.1f}%"})

summary = pd.DataFrame(rows)

if HAS_JINJA2:
    def _style_missing(val):
        if val == "COLUMN NOT FOUND": return "background-color: #faa; font-weight: bold"
        if isinstance(val, int) and val > 0: return "background-color: #fdd"
        return ""
    display(summary.style.map(_style_missing, subset=["Missing count"]))
else:
    display(summary)

## 2 · Distinct value tables
Each table shows every unique value with its count and a flag for anything suspicious.

### 2a · Source File
Expected: a single consistent PDF filename.

In [ ]:
col = "Source File"
EXPECTED_SOURCE = "Ceiling 2 DEC.pdf"

tbl = df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="Count")

def flag_source(v):
    if pd.isna(v) or v == "": return "⚠ blank"
    if v != EXPECTED_SOURCE:   return "⚠ unexpected filename"
    return ""

tbl["Flag"] = tbl[col].apply(flag_source)
show(tbl)

### 2b · Page
Expected: integers 1–24 (one per timesheet page).

In [ ]:
col = "Page"
MAX_PAGE = 24

tbl = df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="Count")

def flag_page(v):
    if pd.isna(v) or v == "": return "⚠ blank"
    try:
        n = int(v)
        if n < 1:        return "⚠ zero or negative"
        if n > MAX_PAGE: return f"⚠ exceeds max page ({MAX_PAGE})"
        return ""
    except ValueError:   return "⚠ non-integer"

tbl["Flag"] = tbl[col].apply(flag_page)
tbl["_sort"] = pd.to_numeric(tbl[col], errors="coerce")
tbl = tbl.sort_values("_sort").drop(columns="_sort")
show(tbl)

### 2c · Date
Expected: December 2025 dates in MM/DD/YY format.

Known issues: mixed formats (`12-03-25`, `12.09.25`), wrong years (`10/08/05`, `12/10/2015`, `12/12/85`), impossible date (`14/19/2025`).

In [ ]:
col = "Date"
EXPECTED_YEAR = 2025

tbl = df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="Count")

def flag_date(v):
    if pd.isna(v) or v == "": return "⚠ blank"
    try:
        parsed = pd.to_datetime(str(v), dayfirst=False)
        if parsed.year != EXPECTED_YEAR:
            return f"⚠ wrong year ({parsed.year})"
        if not re.match(r"^\d{1,2}/\d{1,2}/\d{2}$", str(v)):
            return "⚠ non-standard format"
        return ""
    except Exception:
        return "⚠ unparseable"

tbl["Flag"] = tbl[col].apply(flag_date)
show(tbl)

### 2d · Day
Expected: English weekday names.

Known issues: Spanish names (`MARTES`, `JUEVES`, `VIERNES`, `Miércoles`), the literal string `Today`, and 14 blanks.

In [ ]:
col = "Day"
VALID_DAYS  = {"Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"}
SPANISH_MAP = {
    "LUNES": "Monday", "MARTES": "Tuesday", "MIERCOLES": "Wednesday",
    "MIÉRCOLES": "Wednesday", "JUEVES": "Thursday", "VIERNES": "Friday",
    "SABADO": "Saturday", "SÁBADO": "Saturday", "DOMINGO": "Sunday"
}

tbl = df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="Count")

def flag_day(v):
    if pd.isna(v) or v == "": return "⚠ blank"
    if v.title() in VALID_DAYS:  return ""
    if v.upper() in SPANISH_MAP: return f"⚠ Spanish — suggest '{SPANISH_MAP[v.upper()]}'"
    return "⚠ unexpected value"

tbl["Flag"] = tbl[col].apply(flag_day)
show(tbl)

### 2e · Title
Expected: `SP Cleaner`, `Access Control`, `Supervisor`, `Safety`.

Known issues: `A` (11 rows), `SP` (4 rows), `Safety Mgr` (1 row) — likely abbreviations or OCR errors.

In [ ]:
col = "Title"
VALID_TITLES = {"SP Cleaner", "Access Control", "Supervisor", "Safety"}

tbl = df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="Count")

def flag_title(v):
    if pd.isna(v) or v == "": return "⚠ blank"
    if v in VALID_TITLES:      return ""
    return "⚠ review — not in known titles"

tbl["Flag"] = tbl[col].apply(flag_title)
show(tbl)

### 2f · Employee Name
Expected: `Firstname Lastname`. Flags blanks (15 rows), single-word names, all-caps, and names containing digits.

In [ ]:
col = "Employee Name"

tbl = df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="Count")

def flag_name(v):
    if pd.isna(v) or v == "": return "⚠ blank"
    if any(c.isdigit() for c in v):   return "⚠ contains digit"
    if len(v) > 2 and v == v.upper(): return "⚠ all caps"
    if len(v.split()) < 2:            return "⚠ single word — missing last name?"
    return ""

tbl["Flag"] = tbl[col].apply(flag_name)
show(tbl)

### 2g · EIN
Expected: 9-digit numeric strings.

Known issues: 31 blanks, multiple 8-digit EINs (e.g. `21868101`) that appear to be missing a leading digit.

In [ ]:
col = "EIN"

tbl = df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="Count")

def flag_ein(v):
    if pd.isna(v) or v == "": return "⚠ blank"
    clean = str(v).replace("-", "").replace(" ", "")
    if not clean.isdigit():   return "⚠ non-numeric"
    if len(clean) != 9:       return f"⚠ length {len(clean)} (expected 9)"
    return ""

tbl["Flag"] = tbl[col].apply(flag_ein)
tbl["_sort"] = pd.to_numeric(tbl[col], errors="coerce")
tbl = tbl.sort_values("_sort").drop(columns="_sort")
show(tbl)

### 2h · Actual Hours
Expected: numeric values between 0 and 24.

Known issues: 123 blanks (absent/out employees). Non-blank values range from 5.5 to 11.5.

In [ ]:
col = "Actual Hours"

tbl = df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="Count")

def flag_hours(v):
    if pd.isna(v) or v == "": return "⚠ blank"
    try:
        n = float(v)
        if n < 0:  return "⚠ negative"
        if n > 24: return "⚠ exceeds 24h"
        return ""
    except ValueError: return "⚠ non-numeric"

tbl["Flag"] = tbl[col].apply(flag_hours)
tbl["_sort"] = pd.to_numeric(tbl[col], errors="coerce")
tbl = tbl.sort_values("_sort", na_position="last").drop(columns="_sort")
show(tbl)

## 3 · Flagged rows — full record view
Every row with at least one issue, with flag reasons shown inline.

In [ ]:
flag_funcs = {
    "Source File":   flag_source,
    "Page":          flag_page,
    "Date":          flag_date,
    "Day":           flag_day,
    "Title":         flag_title,
    "Employee Name": flag_name,
    "EIN":           flag_ein,
    "Actual Hours":  flag_hours,
}

flagged = df.copy()
for col, fn in flag_funcs.items():
    if col in flagged.columns:
        flagged[f"FLAG: {col}"] = flagged[col].apply(fn)

flag_cols    = [c for c in flagged.columns if c.startswith("FLAG: ")]
has_flag     = flagged[flag_cols].apply(lambda r: r.ne("").any(), axis=1)
flagged      = flagged[has_flag].reset_index(drop=True)
display_cols = [c for c in COLUMNS_TO_CHECK if c in flagged.columns] + flag_cols

print(f"Flagged: {len(flagged):,} of {len(df):,} rows ({len(flagged)/len(df)*100:.1f}%)")
display(flagged[display_cols])

## 4 · Export flagged rows for manual review

In [ ]:
out = "secai_flagged_rows.xlsx"
flagged[display_cols].to_excel(out, index=False)
print(f"Saved -> {out}")